In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import os

# This is the safest way on Windows
local_path = r"C:\Users\MSI\Desktop\Thesis\Phi-3mini"

# Convert to absolute path and normalize
local_path = os.path.abspath(local_path)
print(f"Using absolute path: {local_path}")

# Verify the folder and files
if os.path.exists(local_path):
    print("✅ Folder exists!")
    files = os.listdir(local_path)
    print(f"Total files: {len(files)}")
    important = [f for f in files if any(x in f.lower() for x in ['model', 'config', 'tokenizer', 'phi3'])]
    print("Important files found:", important)
else:
    print("❌ Folder not found. Check the path above.")

print("\n" + "="*80)

# Load with extra safety
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    local_path,
    trust_remote_code=True,
    local_files_only=True
)

from transformers import AutoConfig

config = AutoConfig.from_pretrained(
    local_path,
    trust_remote_code=True,
    local_files_only=True
)

print(config)
print(type(config))

print("Loading model... (expect 30-90 seconds on CPU)")
model = AutoModelForCausalLM.from_pretrained(
    local_path,
    trust_remote_code=True,
    #device_map="cpu",
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True,
    local_files_only=True
)

print("\n🎉 SUCCESS! Phi-3 Mini is loaded from local folder.")

# Show the architecture
config = model.config
print("\n" + "="*80)
print("=== Phi-3 Mini Architecture Summary ===")
print(f"Hidden size (d_model)          : {config.hidden_size}")
print(f"Number of Transformer layers   : {config.num_hidden_layers}")
print(f"MLP intermediate size          : {config.intermediate_size}")
print(f"Attention heads                : {config.num_attention_heads}")
print(f"Key-Value heads (GQA)          : {getattr(config, 'num_key_value_heads', config.num_attention_heads)}")
print(f"Vocab size                     : {config.vocab_size}")
print(f"Max context length             : {config.max_position_embeddings} tokens")
print(f"Activation                     : {config.hidden_act}")
print(f"Position encoding              : RoPE")
print(f"Normalization                  : RMSNorm")

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Using absolute path: C:\Users\MSI\Desktop\Thesis\Phi-3mini
✅ Folder exists!
Total files: 10
Important files found: ['config.json', 'configuration_phi3.py', 'model-00001-of-00002.safetensors', 'model-00002-of-00002.safetensors', 'model.safetensors.index.json', 'modeling_phi3.py', 'tokenizer.json', 'tokenizer.model', 'tokenizer_config.json']

Loading tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


Phi3Config {
  "architectures": [
    "Phi3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "auto_map": {
    "AutoConfig": "configuration_phi3.Phi3Config",
    "AutoModelForCausalLM": "modeling_phi3.Phi3ForCausalLM"
  },
  "bos_token_id": 1,
  "dtype": "bfloat16",
  "embd_pdrop": 0.0,
  "eos_token_id": 32000,
  "hidden_act": "silu",
  "hidden_size": 3072,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 131072,
  "model_type": "phi3",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "original_max_position_embeddings": 4096,
  "pad_token_id": 32000,
  "resid_pdrop": 0.0,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "long_factor": [
      1.0700000524520874,
      1.1200000047683716,
      1.149999976158142,
      1.4199999570846558,
      1.5699999332427979,
      1.7999999523162842,
      2.129999876022339,
      2.129999876022339,
      3.009999990463257,
      5.9100003242492

`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

The Model has a Hidden size of 3072 and it has 32 Transformer Layers and it has 8192 Multi Layer Perceptron Intermediate as a sie and it has 32 Attention Heads and 32 key -value heads (GQA) and it's Vocab Size it's 32064 and it's Max Context Length of Tokens is 131072 It's Position Encoding is Rotary Positional Encoding

In [ ]:
print("\n" + "="*80)
print("=== Detailed Parameter Analysis ===")

hidden_size = config.hidden_size
num_layers = config.num_hidden_layers
intermediate_size = config.intermediate_size
vocab_size = config.vocab_size

# 🔹 Embedding parameters
embedding_params = vocab_size * hidden_size

# 🔹 Attention parameters per layer (Q, K, V, Output)
attention_params_per_layer = 4 * (hidden_size * hidden_size)

# 🔹 MLP parameters per layer
mlp_params_per_layer = (hidden_size * intermediate_size) + (intermediate_size * hidden_size)

# 🔹 Total per transformer block
params_per_block = attention_params_per_layer + mlp_params_per_layer

# 🔹 Total transformer parameters
total_transformer_params = params_per_block * num_layers

# 🔹 Total model params (approx)
total_params = total_transformer_params + embedding_params

print(f"Embedding parameters           : {embedding_params:,}")
print(f"Attention params per layer     : {attention_params_per_layer:,}")
print(f"MLP params per layer           : {mlp_params_per_layer:,}")
print(f"Total params per block         : {params_per_block:,}")
print(f"Total transformer params       : {total_transformer_params:,}")
print(f"Estimated total parameters     : {total_params:,}")